# Transformer Forecasting Notebook Overview

This notebook builds a Transformer-based workflow for probabilistic electricity load forecasting. It loads and preprocesses hourly demand data, creates time-based features, trains and tunes Transformer models, and evaluates multi-step quantile forecasts. It also saves model predictions, tuning results, ablation results, and loss curves for later analysis.


Imports the core data-processing, scaling, and PyTorch dependencies used throughout the Transformer forecasting workflow.


In [1]:
# Import the required dependencies.
import pandas as pd
import numpy as np
import zipfile
import requests
import io
from sklearn.preprocessing import StandardScaler

import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.nn as nn
import math


# Utility Notebook Setup


Preserves an optional cleanup command for resetting a Colab-style project folder when needed.


In [2]:
# !rm -rf deep_learning_project

Preserves an optional repository clone command for recreating the project workspace when needed.


In [3]:
# !git clone https://github.com/samarthsingh1/deep_learning_project

Preserves optional notebook conversion and execution commands for loading shared data utility code.


In [4]:
# !jupyter nbconvert --to script deep_learning_project/experimentation/data_utils.ipynb
# %run deep_learning_project/experimentation/data_utils.py

Defines the data-loading routine that downloads the UCI electricity load dataset, extracts it in memory, and reads it into a dataframe.


In [5]:
def load_electricity_data():
  # Download the compressed electricity dataset.
  url = "https://archive.ics.uci.edu/static/public/321/electricityloaddiagrams20112014.zip"
  response = requests.get(url)

  # Extract the archive in memory.
  z = zipfile.ZipFile(io.BytesIO(response.content))

  # Read the text file contained in the archive.
  file_name = z.namelist()[0]   # This should be LD2011_2014.txt.

  df = pd.read_csv(
      z.open(file_name),
      sep=";",
      decimal=",",
      parse_dates=[0],
      quotechar='"'
  )

  # Standardize the datetime column.
  df.rename(columns={df.columns[0]: "datetime"}, inplace=True)
  df.set_index("datetime", inplace=True)
  return df

Loads the raw electricity dataset for the preprocessing pipeline.


In [6]:
df = load_electricity_data()

Aggregates customer-level load columns into a single total-load series and resamples it to hourly totals.


In [7]:
def aggregate_data(df):
  df["total_load"] = df.sum(axis=1)
  df_hourly = df["total_load"].resample("H").sum().to_frame()
  df_hourly.head()
  return df_hourly
df_hourly = aggregate_data(df)

/tmp/ipykernel_4559/3915923716.py:3: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df["total_load"].resample("H").sum().to_frame()


Constructs calendar and cyclical time features to represent hourly, weekly, and monthly seasonality.


In [8]:
def create_features(df_hourly):
  # Convert a Series input to a DataFrame before feature creation.
  if isinstance(df_hourly, pd.Series):
      df_hourly = df_hourly.to_frame(name="total_load")

  # Ensure the index uses datetime values.
  df_hourly.index = pd.to_datetime(df_hourly.index)

  # Copy the data to preserve the original input.
  df_features = df_hourly.copy()

  # Create calendar-based time features.
  df_features["hour_of_day"] = df_features.index.hour
  df_features["day_of_week"] = df_features.index.dayofweek   # Monday is 0 and Sunday is 6.
  df_features["is_weekend"] = (df_features["day_of_week"] >= 5).astype(int)
  df_features["month"] = df_features.index.month

  # Add cyclical encodings for recurring time patterns.
  df_features["hour_sin"] = np.sin(2 * np.pi * df_features["hour_of_day"] / 24)
  df_features["hour_cos"] = np.cos(2 * np.pi * df_features["hour_of_day"] / 24)

  df_features["dayofweek_sin"] = np.sin(2 * np.pi * df_features["day_of_week"] / 7)
  df_features["dayofweek_cos"] = np.cos(2 * np.pi * df_features["day_of_week"] / 7)

  df_features["month_sin"] = np.sin(2 * np.pi * (df_features["month"] - 1) / 12)
  df_features["month_cos"] = np.cos(2 * np.pi * (df_features["month"] - 1) / 12)

  return df_features

Creates the feature-enriched dataframe and displays a brief preview of the resulting rows and columns.


In [9]:
df_features = create_features(df_hourly)
print(df_features.head())
print(df_features.columns)

                        total_load  hour_of_day  day_of_week  is_weekend  \
datetime                                                                   
2011-01-01 00:00:00  207058.270272            0            5           1   
2011-01-01 01:00:00  265378.510747            1            5           1   
2011-01-01 02:00:00  263924.219533            2            5           1   
2011-01-01 03:00:00  266306.134264            3            5           1   
2011-01-01 04:00:00  259854.210701            4            5           1   

                     month  hour_sin  hour_cos  dayofweek_sin  dayofweek_cos  \
datetime                                                                       
2011-01-01 00:00:00      1  0.000000  1.000000      -0.974928      -0.222521   
2011-01-01 01:00:00      1  0.258819  0.965926      -0.974928      -0.222521   
2011-01-01 02:00:00      1  0.500000  0.866025      -0.974928      -0.222521   
2011-01-01 03:00:00      1  0.707107  0.707107      -0.974928      

Defines an IQR-based diagnostic helper for identifying unusually high or low target-load observations.


In [10]:
def check_outliers_iqr(df, target_col="total_load"):
    """
    Checks outliers in a numeric column using IQR rule.
    Returns summary dictionary and dataframe of outlier rows.
    """
    series = df[target_col].dropna()

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_mask = (series < lower_bound) | (series > upper_bound)
    outliers = df.loc[outlier_mask]

    summary = {
        "target_col": target_col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "num_outliers": int(outlier_mask.sum()),
        "pct_outliers": round(100 * outlier_mask.mean(), 2)
    }

    return summary, outliers
summary, outliers = check_outliers_iqr(df_hourly, target_col="total_load")
print(summary)
print(outliers.head())

{'target_col': 'total_load', 'q1': np.float64(525816.5873144441), 'q3': np.float64(992780.6161335747), 'iqr': np.float64(466964.0288191306), 'lower_bound': np.float64(-174629.45591425174), 'upper_bound': np.float64(1693226.6593622705), 'num_outliers': 84, 'pct_outliers': np.float64(0.24)}
                       total_load
datetime                         
2012-07-17 16:00:00  1.697254e+06
2012-07-17 17:00:00  1.694787e+06
2012-07-18 14:00:00  1.711057e+06
2012-07-21 16:00:00  1.694772e+06
2012-08-09 16:00:00  1.709171e+06


Defines a chronological train, validation, and test split to preserve temporal order and prevent leakage.


In [11]:
def split_time_series(df, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
    """
    Splits dataframe chronologically into train, val, test.
    Ratios must sum to 1.
    """
    if not np.isclose(train_ratio + val_ratio + test_ratio, 1.0):
        raise ValueError("train_ratio + val_ratio + test_ratio must equal 1.0")

    n = len(df)
    train_end = int(n * train_ratio)
    val_end = train_end + int(n * val_ratio)

    train_df = df.iloc[:train_end].copy()
    val_df = df.iloc[train_end:val_end].copy()
    test_df = df.iloc[val_end:].copy()

    return train_df, val_df, test_df

Applies the chronological split to the engineered feature dataframe.


In [12]:
train_df, val_df, test_df = split_time_series(df_features)

Defines train-fitted standardization so validation and test data are scaled without future-data leakage.


In [13]:
def normalize_splits(train_df, val_df, test_df, feature_cols):
    """
    Fits StandardScaler on train_df[feature_cols] only,
    then transforms train/val/test using the same scaler.

    Returns scaled dataframes and fitted scaler.
    """
    scaler = StandardScaler()

    train_scaled = train_df.copy()
    val_scaled = val_df.copy()
    test_scaled = test_df.copy()

    train_scaled[feature_cols] = scaler.fit_transform(train_df[feature_cols])
    val_scaled[feature_cols] = scaler.transform(val_df[feature_cols])
    test_scaled[feature_cols] = scaler.transform(test_df[feature_cols])

    return train_scaled, val_scaled, test_scaled, scaler

Scales the target-load column across the train, validation, and test splits using the train-fitted scaler.


In [14]:
feature_cols_to_scale = [
    "total_load"
]
train_scaled, val_scaled, test_scaled, scaler = normalize_splits(
    train_df, val_df, test_df, feature_cols_to_scale
)

Displays the first rows of the scaled training data for a quick sanity check.


In [15]:
train_scaled.head()

,total_load,hour_of_day,day_of_week,is_weekend,month,hour_sin,hour_cos,dayofweek_sin,dayofweek_cos,month_sin,month_cos
datetime,,,,,,,,,,,
2011-01-01 00:00:00,-1.577495,0,5,1,1,0.000000,1.000000,-0.974928,-0.222521,0.0,1.0
2011-01-01 01:00:00,-1.404373,1,5,1,1,0.258819,0.965926,-0.974928,-0.222521,0.0,1.0
2011-01-01 02:00:00,-1.408690,2,5,1,1,0.500000,0.866025,-0.974928,-0.222521,0.0,1.0
2011-01-01 03:00:00,-1.401619,3,5,1,1,0.707107,0.707107,-0.974928,-0.222521,0.0,1.0
2011-01-01 04:00:00,-1.420772,4,5,1,1,0.866025,0.500000,-0.974928,-0.222521,0.0,1.0


Assigns the scaled splits to concise variable names used by the sequence-building and modeling steps.


In [ ]:
train=train_scaled
val=val_scaled
test=test_scaled
scaler=scaler

Defines sliding-window sequence generation using 168 hours of history to forecast the next 24 hours.


In [17]:
def create_sequences(data, input_window=168, forecast_horizon=24):
    feature_cols = ['total_load', 'hour_sin', 'hour_cos',
                    'dayofweek_sin', 'dayofweek_cos',
                    'month_sin', 'month_cos']

    features = data[feature_cols].values
    targets = data['total_load'].values

    X, y = [], []
    for i in range(len(features) - input_window - forecast_horizon + 1):
        X.append(features[i : i + input_window])
        y.append(targets[i + input_window : i + input_window + forecast_horizon])

    return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(y), dtype=torch.float32)

Creates train, validation, and test tensors for model training and evaluation.


In [18]:
X_train, y_train = create_sequences(train)
X_val, y_val = create_sequences(val)
X_test, y_test = create_sequences(test)

# print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
# print(f"X_val:   {X_val.shape},   y_val:   {y_val.shape}")
# print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")

Defines PyTorch dataloaders for batched training, validation, and testing, including a batch-shape sanity check.


In [19]:


def create_dataloaders(X_train, y_train, X_val, y_val, X_test, y_test, batch_size=64):
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

    # Verify the first batch dimensions.
    x_batch, y_batch = next(iter(train_loader))
    print(f"Input batch shape:  {x_batch.shape}")
    print(f"Target batch shape: {y_batch.shape}")

    return train_loader, val_loader, test_loader



Builds dataloaders from the prepared sequence tensors.


In [20]:
train_loader, val_loader, test_loader = create_dataloaders(X_train, y_train, X_val, y_val, X_test, y_test)

Input batch shape:  torch.Size([64, 168, 7])
Target batch shape: torch.Size([64, 24])


# Transformer

Defines sinusoidal positional encoding so the Transformer can use temporal ordering within each input window.


In [21]:


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # Shape: [1, max_len, d_model].

        self.register_buffer('pe', pe)

    def forward(self, x):
        # Input shape: [batch, seq_len, d_model].
        x = x + self.pe[:, :x.size(1), :]
        return x

Defines the electricity forecasting Transformer architecture for producing multi-step quantile predictions.


In [22]:
# Remove or skip any earlier cell that defines ElectricityTransformer.

class ElectricityTransformer(nn.Module):
    def __init__(self, input_features=7, d_model=64, nhead=4, num_layers=2,
                 dim_feedforward=128, dropout=0.1, input_window=168,
                 forecast_horizon=24, num_quantiles=3):
        super().__init__()
        self.forecast_horizon = forecast_horizon
        self.num_quantiles = num_quantiles

        self.input_projection = nn.Linear(input_features, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_len=input_window)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.output_head = nn.Sequential(
            nn.Linear(d_model * input_window, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, forecast_horizon * num_quantiles)
        )

    def forward(self, x):
        x = self.input_projection(x)
        x = self.positional_encoding(x)
        x = self.transformer_encoder(x)
        x = x.reshape(x.size(0), -1)
        x = self.output_head(x)
        x = x.reshape(-1, self.forecast_horizon, self.num_quantiles)
        return x



Defines quantile loss for probabilistic forecasting across the selected quantile levels.


In [23]:
class QuantileLoss(nn.Module):
    def __init__(self, quantiles=[0.1, 0.5, 0.9]):
        super().__init__()
        self.quantiles = quantiles

    def forward(self, predictions, targets):
        # Prediction shape: [batch, 24, num_quantiles].
        # Target shape: [batch, 24].

        targets = targets.unsqueeze(-1)  # Reshape to [batch, 24, 1] for broadcasting.
        losses = []

        for i, q in enumerate(self.quantiles):
            errors = targets - predictions[:, :, i].unsqueeze(-1)
            loss = torch.max(q * errors, (q - 1) * errors)
            losses.append(loss.mean())

        return sum(losses) / len(losses)

Defines the training loop with validation tracking, learning-rate scheduling, and best-model checkpointing.


In [24]:
def train_model(model, train_loader, val_loader, criterion, num_epochs=30, lr=1e-3, device='cpu'):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

    train_losses = []
    val_losses = []
    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        # Run the training phase.
        model.train()
        epoch_train_loss = 0
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            predictions = model(x_batch)
            loss = criterion(predictions, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_train_loss += loss.item()

        avg_train_loss = epoch_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Run the validation phase.
        model.eval()
        epoch_val_loss = 0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                predictions = model(x_batch)
                loss = criterion(predictions, y_batch)
                epoch_val_loss += loss.item()

        avg_val_loss = epoch_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        # Update the learning rate based on validation loss.
        scheduler.step(avg_val_loss)

        # Save the best model checkpoint.
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_model.pt')

        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | LR: {current_lr:.6f}")

    # Restore the best model checkpoint.
    model.load_state_dict(torch.load('best_model.pt'))
    print(f"\nTraining complete. Best Val Loss: {best_val_loss:.4f}")

    return model, train_losses, val_losses

Selects the compute device, initializes the baseline Transformer and quantile loss, and runs model training.


In [25]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = ElectricityTransformer().to(device)
criterion = QuantileLoss(quantiles=[0.1, 0.5, 0.9])

model, train_losses, val_losses = train_model(model, train_loader, val_loader, criterion, num_epochs=30, device=device)

Using device: cuda
Epoch 1/30 | Train Loss: 0.0791 | Val Loss: 0.0385 | LR: 0.001000
Epoch 2/30 | Train Loss: 0.0498 | Val Loss: 0.0381 | LR: 0.001000
Epoch 3/30 | Train Loss: 0.0456 | Val Loss: 0.0338 | LR: 0.001000
Epoch 4/30 | Train Loss: 0.0438 | Val Loss: 0.0339 | LR: 0.001000
Epoch 5/30 | Train Loss: 0.0432 | Val Loss: 0.0334 | LR: 0.001000
Epoch 6/30 | Train Loss: 0.0419 | Val Loss: 0.0297 | LR: 0.001000
Epoch 7/30 | Train Loss: 0.0415 | Val Loss: 0.0328 | LR: 0.001000
Epoch 8/30 | Train Loss: 0.0409 | Val Loss: 0.0299 | LR: 0.001000
Epoch 9/30 | Train Loss: 0.0404 | Val Loss: 0.0294 | LR: 0.001000
Epoch 10/30 | Train Loss: 0.0402 | Val Loss: 0.0285 | LR: 0.001000
Epoch 11/30 | Train Loss: 0.0393 | Val Loss: 0.0276 | LR: 0.001000
Epoch 12/30 | Train Loss: 0.0393 | Val Loss: 0.0279 | LR: 0.001000
Epoch 13/30 | Train Loss: 0.0390 | Val Loss: 0.0267 | LR: 0.001000
Epoch 14/30 | Train Loss: 0.0389 | Val Loss: 0.0283 | LR: 0.001000
Epoch 15/30 | Train Loss: 0.0385 | Val Loss: 0.0290 

Runs test-set inference for the baseline Transformer and collects probabilistic predictions.


In [33]:
# =========================================================
# 8a. Run test-set inference for the base Transformer.
# =========================================================

model.eval()
base_predictions = []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        preds = model(x_batch)
        base_predictions.append(preds.cpu().numpy())

transformer_all_preds = np.concatenate(base_predictions, axis=0)
print(f"Base Transformer predictions shape: {transformer_all_preds.shape}")

Base Transformer predictions shape: (5070, 24, 3)


Runs hyperparameter tuning across Transformer configurations and records validation losses for comparison.


In [26]:
# =========================================================
# 9. Run hyperparameter tuning.
# =========================================================
# Run a systematic grid search over key Transformer hyperparameters.
# Train each configuration for 30 epochs and evaluate it using
# validation quantile loss. Use the best configuration
# to train the final fine-tuned model.
# =========================================================

def train_and_evaluate_config(config, train_loader, val_loader, device, num_epochs=30):
    """
    Train a Transformer with the given config and return best val loss.
    """
    model = ElectricityTransformer(
        input_features=7,
        d_model=config['d_model'],
        nhead=config['nhead'],
        num_layers=config['num_layers'],
        dim_feedforward=config['d_model'] * 2,
        dropout=config['dropout'],
        input_window=168,
        forecast_horizon=24,
        num_quantiles=3
    ).to(device)

    criterion = QuantileLoss(quantiles=[0.1, 0.5, 0.9])
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=3, factor=0.5
    )

    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        # Run the training phase.
        model.train()
        epoch_train_loss = 0
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            preds = model(x_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_train_loss += loss.item()

        avg_train_loss = epoch_train_loss / len(train_loader)

        # Run the validation phase.
        model.eval()
        epoch_val_loss = 0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                preds = model(x_batch)
                loss = criterion(preds, y_batch)
                epoch_val_loss += loss.item()

        avg_val_loss = epoch_val_loss / len(val_loader)
        scheduler.step(avg_val_loss)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss

    return best_val_loss


# =========================================================
# 9a. Define the search space and run tuning.
# =========================================================

configs = [
    {'d_model': 64,  'nhead': 4, 'num_layers': 2, 'dropout': 0.1, 'lr': 1e-3},  # Baseline configuration.
    {'d_model': 128, 'nhead': 4, 'num_layers': 2, 'dropout': 0.1, 'lr': 1e-3},
    {'d_model': 64,  'nhead': 8, 'num_layers': 2, 'dropout': 0.1, 'lr': 1e-3},
    {'d_model': 64,  'nhead': 4, 'num_layers': 3, 'dropout': 0.1, 'lr': 1e-3},
    {'d_model': 128, 'nhead': 8, 'num_layers': 3, 'dropout': 0.1, 'lr': 1e-3},
    {'d_model': 64,  'nhead': 4, 'num_layers': 2, 'dropout': 0.2, 'lr': 1e-3},
    {'d_model': 128, 'nhead': 4, 'num_layers': 2, 'dropout': 0.1, 'lr': 5e-4},
    {'d_model': 128, 'nhead': 8, 'num_layers': 2, 'dropout': 0.1, 'lr': 1e-3},
]

print("=" * 80)
print("  HYPERPARAMETER TUNING — GRID SEARCH")
print("=" * 80)

tuning_results = []
for i, config in enumerate(configs):
    print(f"\nConfig {i+1}/{len(configs)}: {config}")
    best_val = train_and_evaluate_config(config, train_loader, val_loader, device, num_epochs=30)
    tuning_results.append({**config, 'best_val_loss': best_val})
    print(f"  -> Best Val Loss: {best_val:.4f}")

# =========================================================
# 9b. Display the results table.
# =========================================================

import pandas as pd

tuning_df = pd.DataFrame(tuning_results)
tuning_df = tuning_df.sort_values('best_val_loss').reset_index(drop=True)
tuning_df.index = tuning_df.index + 1  # Use one-indexed ranking.
tuning_df.index.name = 'Rank'

print("\n" + "=" * 80)
print("  TUNING RESULTS (sorted by best validation loss)")
print("=" * 80)
print(tuning_df.to_string())
print("=" * 80)

best_config = tuning_results[tuning_df.index[0] - 1]
print(f"\nBest config: {best_config}")

  HYPERPARAMETER TUNING — GRID SEARCH

Config 1/8: {'d_model': 64, 'nhead': 4, 'num_layers': 2, 'dropout': 0.1, 'lr': 0.001}
  -> Best Val Loss: 0.0241

Config 2/8: {'d_model': 128, 'nhead': 4, 'num_layers': 2, 'dropout': 0.1, 'lr': 0.001}
  -> Best Val Loss: 0.0266

Config 3/8: {'d_model': 64, 'nhead': 8, 'num_layers': 2, 'dropout': 0.1, 'lr': 0.001}
  -> Best Val Loss: 0.0241

Config 4/8: {'d_model': 64, 'nhead': 4, 'num_layers': 3, 'dropout': 0.1, 'lr': 0.001}
  -> Best Val Loss: 0.0242

Config 5/8: {'d_model': 128, 'nhead': 8, 'num_layers': 3, 'dropout': 0.1, 'lr': 0.001}
  -> Best Val Loss: 0.0278

Config 6/8: {'d_model': 64, 'nhead': 4, 'num_layers': 2, 'dropout': 0.2, 'lr': 0.001}
  -> Best Val Loss: 0.0296

Config 7/8: {'d_model': 128, 'nhead': 4, 'num_layers': 2, 'dropout': 0.1, 'lr': 0.0005}
  -> Best Val Loss: 0.0249

Config 8/8: {'d_model': 128, 'nhead': 8, 'num_layers': 2, 'dropout': 0.1, 'lr': 0.001}
  -> Best Val Loss: 0.0258

  TUNING RESULTS (sorted by best validation 

Retrains the Transformer with the best hyperparameter configuration and saves the fine-tuned weights.


In [27]:
# =========================================================
# 10. Train the fine-tuned Transformer.
# =========================================================
# Retrain the Transformer using the best hyperparameter
# configuration from the grid search.
# Use this as the final production model.
# Save the weights as 'best_model_finetuned.pt'.
# =========================================================

best_hp = tuning_df.iloc[0]
print("Fine-tuning with best config:")
print(f"  d_model:    {int(best_hp['d_model'])}")
print(f"  nhead:      {int(best_hp['nhead'])}")
print(f"  num_layers: {int(best_hp['num_layers'])}")
print(f"  dropout:    {best_hp['dropout']}")
print(f"  lr:         {best_hp['lr']}")
print()

model_finetuned = ElectricityTransformer(
    input_features=7,
    d_model=int(best_hp['d_model']),
    nhead=int(best_hp['nhead']),
    num_layers=int(best_hp['num_layers']),
    dim_feedforward=int(best_hp['d_model']) * 2,
    dropout=best_hp['dropout'],
    input_window=168,
    forecast_horizon=24,
    num_quantiles=3
).to(device)

criterion_ft = QuantileLoss(quantiles=[0.1, 0.5, 0.9])

model_finetuned, ft_train_losses, ft_val_losses = train_model(
    model_finetuned,
    train_loader,
    val_loader,
    criterion_ft,
    num_epochs=30,
    lr=best_hp['lr'],
    device=device
)

# =========================================================
# 10a. Save the fine-tuned model.
# =========================================================

torch.save(model_finetuned.state_dict(), 'best_model_finetuned.pt')
print(f"\nFine-tuned model saved as 'best_model_finetuned.pt'")

Fine-tuning with best config:
  d_model:    64
  nhead:      8
  num_layers: 2
  dropout:    0.1
  lr:         0.001

Epoch 1/30 | Train Loss: 0.0812 | Val Loss: 0.0431 | LR: 0.001000
Epoch 2/30 | Train Loss: 0.0505 | Val Loss: 0.0353 | LR: 0.001000
Epoch 3/30 | Train Loss: 0.0479 | Val Loss: 0.0324 | LR: 0.001000
Epoch 4/30 | Train Loss: 0.0458 | Val Loss: 0.0350 | LR: 0.001000
Epoch 5/30 | Train Loss: 0.0444 | Val Loss: 0.0334 | LR: 0.001000
Epoch 6/30 | Train Loss: 0.0432 | Val Loss: 0.0310 | LR: 0.001000
Epoch 7/30 | Train Loss: 0.0430 | Val Loss: 0.0302 | LR: 0.001000
Epoch 8/30 | Train Loss: 0.0419 | Val Loss: 0.0320 | LR: 0.001000
Epoch 9/30 | Train Loss: 0.0416 | Val Loss: 0.0311 | LR: 0.001000
Epoch 10/30 | Train Loss: 0.0413 | Val Loss: 0.0298 | LR: 0.001000
Epoch 11/30 | Train Loss: 0.0409 | Val Loss: 0.0279 | LR: 0.001000
Epoch 12/30 | Train Loss: 0.0403 | Val Loss: 0.0325 | LR: 0.001000
Epoch 13/30 | Train Loss: 0.0401 | Val Loss: 0.0281 | LR: 0.001000
Epoch 14/30 | Train 

Updates the configuration training helper so ablation experiments can vary the number of input features.


In [28]:
def train_and_evaluate_config(config, train_loader, val_loader, device, num_epochs=30):
    """
    Train a Transformer with the given config and return best val loss.
    Supports variable input_features for ablation studies.
    """
    input_feat = config.get('input_features', 7)

    model = ElectricityTransformer(
        input_features=input_feat,
        d_model=config['d_model'],
        nhead=config['nhead'],
        num_layers=config['num_layers'],
        dim_feedforward=config['d_model'] * 2,
        dropout=config['dropout'],
        input_window=168,
        forecast_horizon=24,
        num_quantiles=3
    ).to(device)

    criterion = QuantileLoss(quantiles=[0.1, 0.5, 0.9])
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=3, factor=0.5
    )

    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        model.train()
        epoch_train_loss = 0
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            preds = model(x_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_train_loss += loss.item()

        avg_train_loss = epoch_train_loss / len(train_loader)

        model.eval()
        epoch_val_loss = 0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                preds = model(x_batch)
                loss = criterion(preds, y_batch)
                epoch_val_loss += loss.item()

        avg_val_loss = epoch_val_loss / len(val_loader)
        scheduler.step(avg_val_loss)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss

    return best_val_loss

Runs a feature-set ablation study to compare how different temporal feature groups affect validation loss.


In [29]:
# =========================================================
# 11. Run the feature-importance ablation study.
# =========================================================
# Train the Transformer with different feature subsets to
# measure each feature group's contribution.
# Evaluate the following feature subsets:
#   A) Load only with one feature.
#   B) Load plus hour sine and cosine with three features.
#   C) Load plus hour and day sine and cosine with five features.
#   D) All seven features as the baseline.
# Use the best configuration from hyperparameter tuning.
# =========================================================

def create_sequences_subset(data, feature_cols, input_window=168, forecast_horizon=24):
    """Create sequences with a specific subset of features."""
    features = data[feature_cols].values
    targets = data['total_load'].values

    X, y = [], []
    for i in range(len(features) - input_window - forecast_horizon + 1):
        X.append(features[i : i + input_window])
        y.append(targets[i + input_window : i + input_window + forecast_horizon])

    return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(y), dtype=torch.float32)


ablation_configs = {
    'Load only': ['total_load'],
    'Load + Hour': ['total_load', 'hour_sin', 'hour_cos'],
    'Load + Hour + Day': ['total_load', 'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos'],
    'All features': ['total_load', 'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'month_sin', 'month_cos'],
}

print("=" * 80)
print("  ABLATION STUDY — FEATURE IMPORTANCE")
print("=" * 80)

ablation_results = []

for name, feat_cols in ablation_configs.items():
    print(f"\n--- {name} ({len(feat_cols)} features) ---")

    X_tr_ab, y_tr_ab = create_sequences_subset(train, feat_cols)
    X_va_ab, y_va_ab = create_sequences_subset(val, feat_cols)

    tr_loader = DataLoader(TensorDataset(X_tr_ab, y_tr_ab), batch_size=64, shuffle=True)
    va_loader = DataLoader(TensorDataset(X_va_ab, y_va_ab), batch_size=64, shuffle=False)

    best_val = train_and_evaluate_config(
        {
            'd_model': int(best_hp['d_model']),
            'nhead': int(best_hp['nhead']),
            'num_layers': int(best_hp['num_layers']),
            'dropout': best_hp['dropout'],
            'lr': best_hp['lr'],
            'input_features': len(feat_cols),
        },
        tr_loader, va_loader, device, num_epochs=30
    )

    ablation_results.append({
        'Feature Set': name,
        'Num Features': len(feat_cols),
        'Best Val Loss': best_val,
    })
    print(f"  -> Best Val Loss: {best_val:.4f}")

ablation_feat_df = pd.DataFrame(ablation_results)
print("\n" + "=" * 80)
print("  FEATURE ABLATION RESULTS")
print("=" * 80)
print(ablation_feat_df.to_string(index=False))
print("=" * 80)

  ABLATION STUDY — FEATURE IMPORTANCE

--- Load only (1 features) ---
  -> Best Val Loss: 0.0261

--- Load + Hour (3 features) ---
  -> Best Val Loss: 0.0241

--- Load + Hour + Day (5 features) ---
  -> Best Val Loss: 0.0253

--- All features (7 features) ---
  -> Best Val Loss: 0.0243

  FEATURE ABLATION RESULTS
      Feature Set  Num Features  Best Val Loss
        Load only             1       0.026082
      Load + Hour             3       0.024140
Load + Hour + Day             5       0.025319
     All features             7       0.024282


Runs an input-window ablation study to compare shorter and longer historical lookback lengths.


In [30]:
# =========================================================
# 12. Run the input-window ablation study.
# =========================================================
# Train the Transformer with different lookback windows
# to assess the seven-day, 168-hour window choice.
# Test 48, 72, 168, and 336-hour lookback windows.
# =========================================================

window_sizes = [48, 72, 168, 336]

print("=" * 80)
print("  ABLATION STUDY — INPUT WINDOW SIZE")
print("=" * 80)

window_results = []

for window in window_sizes:
    print(f"\n--- Window: {window}h ({window//24} days) ---")

    feature_cols_w = ['total_load', 'hour_sin', 'hour_cos',
                      'dayofweek_sin', 'dayofweek_cos',
                      'month_sin', 'month_cos']

    # Create sequences for the current window size.
    def create_seq_window(data, input_window):
        features = data[feature_cols_w].values
        targets = data['total_load'].values
        X, y = [], []
        for i in range(len(features) - input_window - 24 + 1):
            X.append(features[i : i + input_window])
            y.append(targets[i + input_window : i + input_window + 24])
        return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(y), dtype=torch.float32)

    X_tr_w, y_tr_w = create_seq_window(train, window)
    X_va_w, y_va_w = create_seq_window(val, window)

    tr_loader_w = DataLoader(TensorDataset(X_tr_w, y_tr_w), batch_size=64, shuffle=True)
    va_loader_w = DataLoader(TensorDataset(X_va_w, y_va_w), batch_size=64, shuffle=False)

    # Build a model for the current window size.
    model_w = ElectricityTransformer(
        input_features=7,
        d_model=int(best_hp['d_model']),
        nhead=int(best_hp['nhead']),
        num_layers=int(best_hp['num_layers']),
        dim_feedforward=int(best_hp['d_model']) * 2,
        dropout=best_hp['dropout'],
        input_window=window,
        forecast_horizon=24,
        num_quantiles=3
    ).to(device)

    criterion_w = QuantileLoss(quantiles=[0.1, 0.5, 0.9])
    optimizer_w = torch.optim.Adam(model_w.parameters(), lr=best_hp['lr'])
    scheduler_w = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_w, mode='min', patience=3, factor=0.5)

    best_val_w = float('inf')
    for epoch in range(30):
        model_w.train()
        for x_batch, y_batch in tr_loader_w:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer_w.zero_grad()
            preds = model_w(x_batch)
            loss = criterion_w(preds, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_w.parameters(), max_norm=1.0)
            optimizer_w.step()

        model_w.eval()
        epoch_val = 0
        with torch.no_grad():
            for x_batch, y_batch in va_loader_w:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                preds = model_w(x_batch)
                loss = criterion_w(preds, y_batch)
                epoch_val += loss.item()
        avg_val = epoch_val / len(va_loader_w)
        scheduler_w.step(avg_val)
        if avg_val < best_val_w:
            best_val_w = avg_val

    window_results.append({
        'Window (hours)': window,
        'Window (days)': window // 24,
        'Best Val Loss': best_val_w,
    })
    print(f"  -> Best Val Loss: {best_val_w:.4f}")

window_df = pd.DataFrame(window_results)
print("\n" + "=" * 80)
print("  WINDOW SIZE ABLATION RESULTS")
print("=" * 80)
print(window_df.to_string(index=False))
print("=" * 80)

  ABLATION STUDY — INPUT WINDOW SIZE

--- Window: 48h (2 days) ---
  -> Best Val Loss: 0.0232

--- Window: 72h (3 days) ---
  -> Best Val Loss: 0.0238

--- Window: 168h (7 days) ---
  -> Best Val Loss: 0.0237

--- Window: 336h (14 days) ---
  -> Best Val Loss: 0.0264

  WINDOW SIZE ABLATION RESULTS
 Window (hours)  Window (days)  Best Val Loss
             48              2       0.023219
             72              3       0.023784
            168              7       0.023743
            336             14       0.026367


Runs test-set inference for the fine-tuned Transformer and collects predictions alongside scaled actual values.


In [31]:
# =========================================================
# 10b. Run test-set inference for the fine-tuned Transformer.
# =========================================================

model_finetuned.eval()
ft_predictions = []
ft_actuals = []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        preds = model_finetuned(x_batch)
        ft_predictions.append(preds.cpu().numpy())
        ft_actuals.append(y_batch.numpy())

ft_all_preds = np.concatenate(ft_predictions, axis=0)
ft_actuals_scaled = np.concatenate(ft_actuals, axis=0)

print(f"Fine-Tuned Transformer predictions shape: {ft_all_preds.shape}")
print("Stored variables: ft_all_preds, ft_actuals_scaled")

Fine-Tuned Transformer predictions shape: (5070, 24, 3)
Stored variables: ft_all_preds, ft_actuals_scaled


Creates the predictions directory and saves model outputs, tuning results, ablation results, and loss curves.


In [34]:
# =========================================================
# Save all Transformer predictions.
# =========================================================

import os
os.makedirs('predictions', exist_ok=True)

np.save('predictions/transformer_all_preds.npy', transformer_all_preds)
np.save('predictions/ft_all_preds.npy', ft_all_preds)

# Save tuning and ablation results.
tuning_df.to_csv('predictions/tuning_results.csv')
ablation_feat_df.to_csv('predictions/ablation_features.csv', index=False)
window_df.to_csv('predictions/ablation_windows.csv', index=False)

# Save loss curves.
np.save('predictions/train_losses.npy', train_losses)
np.save('predictions/val_losses.npy', val_losses)
np.save('predictions/ft_train_losses.npy', ft_train_losses)
np.save('predictions/ft_val_losses.npy', ft_val_losses)

print("All transformer predictions saved to predictions/ folder")

All transformer predictions saved to predictions/ folder
